**1. 建立一個叫做 books 的 index**

In [10]:
import os
import requests
from dotenv import load_dotenv
load_dotenv()

ES_API_URL = os.getenv("ES_API_URL")
ES_API_KEY = os.getenv("ES_API_KEY")
header = {
    "Authorization": f"Apikey {ES_API_KEY}",
    "Content-Type": "application/json"
}
data = {
    "mappings": {
        "properties": {
            "id": {
                "type": "keyword"
            },
            "title": {
                "type": "text",
                "analyzer": "ik_max_word",
                "search_analyzer": "ik_smart"
            }
        }
    }
}
response = requests.put(f"{ES_API_URL}/books", headers=header, json=data)
print(response.json())

{'acknowledged': True, 'shards_acknowledged': True, 'index': 'books'}


**2. 將書籍資料透過 Bulk API 匯入剛剛建好的 index**

In [11]:
import os
import time
import requests
from dotenv import load_dotenv
load_dotenv()

ES_API_URL = os.getenv("ES_API_URL")
ES_API_KEY = os.getenv("ES_API_KEY")
header = {
    "Authorization": f"Apikey {ES_API_KEY}",
    "Content-Type": "application/json"
}
with open("../data/processed/books.jsonl", "r", encoding="utf-8") as f:
    data = f.read()
response = requests.put(f"{ES_API_URL}/books/_bulk?pretty", headers=header, data=data)
time.sleep(3)
if response.status_code != 200:
    print("Error:", response.status_code, response.text)
else:
    print("成功將書籍資料匯入 books index")

成功將書籍資料匯入 books index


**3. 測試搜尋書籍**

In [2]:
import os
import json
import requests
from dotenv import load_dotenv
load_dotenv()

ES_API_URL = os.getenv("ES_API_URL")
ES_API_KEY = os.getenv("ES_API_KEY")
header = {
    "Authorization": f"Apikey {ES_API_KEY}",
    "Content-Type": "application/json"
}
data = {
    "query": {
        "simple_query_string": {
            "query": "HTML + 新手"
        }
    }
}
response = requests.get(f"{ES_API_URL}/books/_search?size=10", headers=header, json=data)
print(json.dumps(response.json(), indent=4, ensure_ascii=False))

{
    "took": 7,
    "timed_out": false,
    "_shards": {
        "total": 1,
        "successful": 1,
        "skipped": 0,
        "failed": 0
    },
    "hits": {
        "total": {
            "value": 2,
            "relation": "eq"
        },
        "max_score": 7.9428473,
        "hits": [
            {
                "_index": "books",
                "_id": "990073708790107976",
                "_score": 7.9428473,
                "_source": {
                    "id": "990073708790107976",
                    "title": "HTML/CSS/JavaScript與前端框架的完美結合 : 使用Bootstrap與PWA技術,新手從這開始! /"
                }
            },
            {
                "_index": "books",
                "_id": "9973824007707976",
                "_score": 7.9428473,
                "_source": {
                    "id": "9973824007707976",
                    "title": "HTML / CSS / JavaScript與前端框架的完美結合 : 使用Bootstrap與PWA技術, 新手從這開始! /"
                }
            }
        ]
    }
}
